# Add `is_misdemeanor` and `has_warrant` Columns

Adds two derived columns to checkpoint14 and saves checkpoint15.

- **`is_misdemeanor`**: `True` if every charge in the row classifies as `Misdemeanor` under MA law; `False` if any charge is `Felony`, `Either`, or `Unknown`; `NaN` for rows with no charges.
- **`has_warrant`**: `True` if any individual charge in the original `Charges` field carried a warrant prefix (bench warrant, default warrant, standard warrant, capias, child in need); `False` otherwise; `NaN` for rows with no charges.

**Input:** `data/checkpoints/checkpoint14_standardized_charges.csv`  
**Output:** `data/checkpoints/checkpoint15_misdemeanor_warrant.csv`

### Imports & Paths

In [1]:
import os
import numpy as np
import pandas as pd

from standardize_charges import extract_warrant_type

NOTEBOOK_DIR = os.getcwd()
DATA_DIR = os.path.join(NOTEBOOK_DIR, "..", "..", "data")
CHECKPOINTS_DIR = os.path.join(DATA_DIR, "checkpoints")

IN_PATH  = os.path.join(CHECKPOINTS_DIR, "checkpoint14_standardized_charges.csv")
OUT_PATH = os.path.join(CHECKPOINTS_DIR, "checkpoint15_misdemeanor_warrant.csv")
LOOKUP_PATH = os.path.join(NOTEBOOK_DIR, "unique_charges_standardized.csv")

### Load checkpoint14

In [2]:
df = pd.read_csv(IN_PATH, low_memory=False)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Rows with charges: {df['standardized_charges'].notna().sum()}")
df.head()

Shape: (424273, 18)
Columns: ['Date', 'Type', 'Location', 'Arrested', 'Location Prefix', 'DOB', 'Charges', 'latitude', 'longitude', 'Cleaned Location', 'person_id', 'category_archive', 'Year', 'crime_severity', 'category', 'Age', 'statutes', 'standardized_charges']
Rows with charges: 4542


,Date,Type,Location,Arrested,Location Prefix,DOB,Charges,latitude,longitude,Cleaned Location,person_id,category_archive,Year,crime_severity,category,Age,statutes,standardized_charges
0,2018-01-01 00:01:00,NOISE ORD,3 HARRIMAN ST,Yes,NaN,1974-07-03,A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...,42.718522,-71.148148,3 HARRIMAN ST,20f72481f083d4756c89b98fd499514c5953a8a4c10253...,PUBLIC_DISTURBANCES,2018,Non-Serious,Public Disturbances,43.0,NaN,a&b on family / household member; strangulatio...
1,2018-01-01 00:08:00,LOUD NOISE,1 HARRIMAN ST FL 2,Yes,NaN,1979-01-21,A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...,42.718522,-71.148148,1 HARRIMAN ST,2d5adb553cad8a22fd72d2e390525997c4963bac1b185e...,PUBLIC_DISTURBANCES,2018,Non-Serious,Public Disturbances,38.0,NaN,a&b on family / household member
2,2018-01-01 00:11:00,ALARM/BURG,16 ALLEN ST,No,MATOS,NaN,NaN,42.710782,-71.151911,16 ALLEN ST,NaN,FIRE_AND_ARSON_INCIDENTS,2018,Non-Serious,Fire and Arson Incidents,NaN,NaN,NaN
3,2018-01-01 00:14:00,DISORDERLY,11 SUMMER ST,No,NaN,NaN,NaN,42.711117,-71.153015,11 SUMMER ST,NaN,PUBLIC_DISTURBANCES,2018,Non-Serious,Public Disturbances,NaN,NaN,NaN
4,2018-01-01 00:27:00,EXTRA SURVEIL,57 SPRINGFIELD ST,Yes,WARD SIX CLUB,2002-06-26,A&B DOMESTIC NO 209A IN EFFECT,42.699339,-71.156938,57 SPRINGFIELD ST,41c43f9f4255dee8e2c910e87f2a983e1b13beecf27eac...,PREVENTIVE_POLICING,2018,Non-Serious,Preventive Policing,15.0,NaN,a&b domestic no 209a in effect


### Build charge_class lookup

Load `unique_charges_standardized.csv` and build a dict: `base_charge → charge_class`.

In [3]:
lookup_df = pd.read_csv(LOOKUP_PATH)
charge_class_lookup = dict(zip(lookup_df["base_charge"], lookup_df["charge_class"]))

print(f"Lookup entries: {len(charge_class_lookup)}")
print(f"Classes present: {set(charge_class_lookup.values())}")

# Quick sanity checks
print()
for sample in ["trespass", "murder", "firearm, carry without license", "drug, possess class a"]:
    print(f"  {sample!r} -> {charge_class_lookup.get(sample, 'NOT FOUND')}")

Lookup entries: 394
Classes present: {'Either', 'Misdemeanor', 'Felony'}

  'trespass' -> Misdemeanor
  'murder' -> Felony
  'firearm, carry without license' -> Either
  'drug, possess class a' -> Misdemeanor


### Compute `has_warrant`

Check the original `Charges` column. Split each row's charges by `;` and run `extract_warrant_type()` on each individual charge. If any charge had a warrant prefix, the row is flagged `True`.

In [4]:
def row_has_warrant(raw_charges):
    """Return True if any individual charge in the raw Charges string
    carries a warrant prefix. Returns NaN for rows with no charges."""
    if pd.isna(raw_charges):
        return np.nan
    parts = [p.strip() for p in str(raw_charges).split(";") if p.strip()]
    for part in parts:
        warrant_type, _ = extract_warrant_type(part)
        if warrant_type != "none":
            return True
    return False


df["has_warrant"] = df["Charges"].apply(row_has_warrant)

warrant_counts = df["has_warrant"].value_counts(dropna=False)
print("has_warrant value counts:")
print(warrant_counts)

# Spot-check
print("\nSample warrant rows:")
sample_warrant = df[df["has_warrant"] == True][["Charges", "standardized_charges", "has_warrant"]].head(5)
for _, row in sample_warrant.iterrows():
    print(f"  Charges: {str(row['Charges'])[:100]}")
    print(f"  Standardized: {row['standardized_charges']}")
    print()

has_warrant value counts:
has_warrant
NaN      419264
False      3244
True       1765
Name: count, dtype: int64

Sample warrant rows:
  Charges: DEFAULT WARRANT: DEFACEMENT MALICIOUS WANTON; PROPERTY C266 S126A
  Standardized: defacement malicious wanton property

  Charges: DEFAULT WARRANT: DRUG, POSSESS CLASS B c94C S34;  2/5/2020; QuickSearch Results Page 8 of 14; DEFAUL
  Standardized: drug, possess class b; larceny under $1200

  Charges: SHOPLIFTING BY ASPORTATION c266 S30A; RESISTING ARREST c268 S32B; Warrant Charges: STANDARD WARRANT:
  Standardized: shoplifting by asportation; resisting arrest; b&e nighttime for felony

  Charges: STANDARD WARRANT: UNINSURED MV/TRAILER c90 S34J; STANDARD WARRANT: DRUG, POSSESS CLASS B c94C S34; S
  Standardized: uninsured mv/trailer; drug, possess class b; inspection/sticker, no; juror fail to attend; drug, distribute class b; conspiracy to violate drug law; drug, distribute class b; drug, possess to distrib class b; drug, distribute class b



### Compute `is_misdemeanor`

Split `standardized_charges` by `"; "` and look up each charge's class in the lookup dict. A row is `True` only when **every** charge in the row classifies as `Misdemeanor`. Any `Felony`, `Either`, or `Unknown` charge makes the row `False`. Rows with no charges return `NaN`.

In [5]:
def row_is_misdemeanor(std_charges):
    """Return True if every standardized charge in the row is Misdemeanor.
    Returns False if any charge is Felony, Either, or Unknown.
    Returns NaN for rows with no charges."""
    if pd.isna(std_charges):
        return np.nan
    parts = [p.strip() for p in str(std_charges).split(";") if p.strip()]
    if not parts:
        return np.nan
    for charge in parts:
        cls = charge_class_lookup.get(charge, "Unknown")
        if cls != "Misdemeanor":
            return False
    return True


df["is_misdemeanor"] = df["standardized_charges"].apply(row_is_misdemeanor)

misdemeanor_counts = df["is_misdemeanor"].value_counts(dropna=False)
print("is_misdemeanor value counts:")
print(misdemeanor_counts)

# Spot-check True rows
print("\nSample misdemeanor-only rows:")
sample_misd = df[df["is_misdemeanor"] == True][["Charges", "standardized_charges", "is_misdemeanor"]].head(5)
for _, row in sample_misd.iterrows():
    print(f"  Charges: {str(row['Charges'])[:100]}")
    print(f"  Standardized: {row['standardized_charges']}")
    print()

# Spot-check False rows (not pure misdemeanor)
print("Sample non-misdemeanor rows:")
sample_non = df[df["is_misdemeanor"] == False][["Charges", "standardized_charges", "is_misdemeanor"]].head(5)
for _, row in sample_non.iterrows():
    print(f"  Charges: {str(row['Charges'])[:100]}")
    print(f"  Standardized: {row['standardized_charges']}")
    print()

is_misdemeanor value counts:
is_misdemeanor
NaN      419731
False      2290
True       2252
Name: count, dtype: int64

Sample misdemeanor-only rows:
  Charges: A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PARTNE
  Standardized: a&b on family / household member

  Charges: A&B DOMESTIC NO 209A IN EFFECT
  Standardized: a&b domestic no 209a in effect

  Charges: USE MV WITHOUT AUTHORITY c90 S24; LARCENY UNDER $250 c266 S30
  Standardized: use mv without authority; larceny under $250

  Charges: DRUG, POSSESS CLASS B c94C S34
  Standardized: drug, possess class b

  Charges: DRUG, POSSESS CLASS B c94C S34
  Standardized: drug, possess class b

Sample non-misdemeanor rows:
  Charges: A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PARTNE; STRANGULATION OR SUFFOCATION
  Standardized: a&b on family / household member; strangulation or suffocation

  Charges: WITNESS, INTIMIDATE c268 S13B; FALSE NAME/SS# TO LAW ENFORCEMENT; THREAT TO COMMIT CRIME c275 S2; AS
  Standardized: witness, intimidate; fa

### Diagnose any charges not found in lookup

Identifies standardized charges that appear in the data but are missing from `unique_charges_standardized.csv`.

In [6]:
all_std = (
    df["standardized_charges"]
    .dropna()
    .str.split("; ")
    .explode()
    .str.strip()
    .unique()
)

missing = [c for c in all_std if c and c not in charge_class_lookup]
if missing:
    print(f"{len(missing)} standardized charges not found in lookup:")
    for c in sorted(missing):
        print(f"  {c!r}")
else:
    print("All standardized charges found in lookup.")

All standardized charges found in lookup.


### Verify & summarize

In [7]:
charged_rows = df[df["standardized_charges"].notna()]

print(f"Total rows: {len(df):,}")
print(f"Rows with charges: {len(charged_rows):,}")
print()
print(f"has_warrant = True  : {(df['has_warrant'] == True).sum():,}")
print(f"has_warrant = False : {(df['has_warrant'] == False).sum():,}")
print(f"has_warrant = NaN   : {df['has_warrant'].isna().sum():,}")
print()
print(f"is_misdemeanor = True  : {(df['is_misdemeanor'] == True).sum():,}")
print(f"is_misdemeanor = False : {(df['is_misdemeanor'] == False).sum():,}")
print(f"is_misdemeanor = NaN   : {df['is_misdemeanor'].isna().sum():,}")
print()
print(f"Final shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Total rows: 424,273
Rows with charges: 4,542

has_warrant = True  : 1,765
has_warrant = False : 3,244
has_warrant = NaN   : 419,264

is_misdemeanor = True  : 2,252
is_misdemeanor = False : 2,290
is_misdemeanor = NaN   : 419,731

Final shape: (424273, 20)
Columns: ['Date', 'Type', 'Location', 'Arrested', 'Location Prefix', 'DOB', 'Charges', 'latitude', 'longitude', 'Cleaned Location', 'person_id', 'category_archive', 'Year', 'crime_severity', 'category', 'Age', 'statutes', 'standardized_charges', 'has_warrant', 'is_misdemeanor']


### Save checkpoint15

In [8]:
df.to_csv(OUT_PATH, index=False)
print(f"Saved checkpoint15: {OUT_PATH}")
print(f"Shape: {df.shape}")

Saved checkpoint15: C:\Users\Indel\Documents\gatewayinitiative-lawrencepd\scripts\charges\..\..\data\checkpoints\checkpoint15_misdemeanor_warrant.csv
Shape: (424273, 20)
